# Tenacious-Bench — ORPO Adapter Inference on Colab T4

**Before running anything:**
1. Runtime → Change runtime type → T4 GPU → Save
2. Upload `colab_upload.zip` to the root of your Google Drive

Run cells top to bottom in order.

In [ ]:
# Cell 1 — Upgrade deps before anything else (avoids torchao version conflict)
!pip install torchao --upgrade -q
!pip install peft --upgrade -q
!pip install transformers accelerate safetensors -q
print('Dependencies ready')

In [ ]:
# Cell 2 — Verify GPU
import torch
assert torch.cuda.is_available(), 'No GPU — go to Runtime > Change runtime type > T4 GPU'
print('Device:', torch.cuda.get_device_name(0))
print('VRAM  :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# Cell 3 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 4 — Unzip and fix Windows backslash paths
import zipfile, pathlib, os, shutil

ZIP_PATH    = '/content/drive/MyDrive/colab_upload.zip'
EXTRACT_DIR = '/content/bench'

pathlib.Path(EXTRACT_DIR).mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_DIR)

# Fix backslash-named entries created by PowerShell Compress-Archive
for item in os.listdir(EXTRACT_DIR):
    if '\\' in item:
        src = f'{EXTRACT_DIR}/{item}'
        dst = EXTRACT_DIR + '/' + item.replace('\\', '/')
        pathlib.Path(dst).parent.mkdir(parents=True, exist_ok=True)
        shutil.move(src, dst)

print('Files after extraction:')
for root, dirs, files in os.walk(EXTRACT_DIR):
    for f in files:
        print(' ', os.path.join(root, f))

In [ ]:
# Cell 5 — Confirm paths and task count
import json, pathlib

SCRIPT   = '/content/bench/generate_model_predictions.py'
ADAPTER  = '/content/bench/orpo_qwen3_0_6b_lora_adapter'
HELD_OUT = '/content/bench/held_out'

all_ok = True
for label, p in [('script', SCRIPT), ('adapter', ADAPTER), ('held_out', HELD_OUT)]:
    exists = pathlib.Path(p).exists()
    print(f'  {label}: {"OK" if exists else "MISSING"} — {p}')
    if not exists:
        all_ok = False

task_count = sum(
    1 for f in pathlib.Path(HELD_OUT).glob('*.jsonl')
    for line in f.read_text().splitlines() if line.strip()
) if pathlib.Path(HELD_OUT).exists() else 0
print(f'  held-out tasks: {task_count}')

if not all_ok or task_count == 0:
    raise RuntimeError('Paths missing or no tasks found — check cell 4 output')

In [ ]:
# Cell 6 — Trained predictions (base + LoRA adapter)
!python /content/bench/generate_model_predictions.py \
    --mode trained \
    --adapter-path /content/bench/orpo_qwen3_0_6b_lora_adapter \
    --held-out-dir /content/bench/held_out \
    --output /content/trained_predictions.jsonl

In [ ]:
# Cell 7 — Prompt-only predictions (bare base model, no adapter)
!python /content/bench/generate_model_predictions.py \
    --mode prompt_only \
    --held-out-dir /content/bench/held_out \
    --output /content/prompt_only_predictions.jsonl

In [ ]:
# Cell 8 — Sanity check outputs
import json, pathlib

for fname in ['trained_predictions.jsonl', 'prompt_only_predictions.jsonl']:
    path = pathlib.Path(f'/content/{fname}')
    if not path.exists():
        print(f'MISSING: {fname}')
        continue
    rows = [json.loads(l) for l in path.read_text().splitlines() if l.strip()]
    print(f'{fname}: {len(rows)} predictions')
    if rows:
        r = rows[0]
        print(f'  task_id : {r["task_id"]}')
        print(f'  subject : {r["candidate_output"].get("subject", "")[:80]}')
        print(f'  latency : {r["latency_ms"]} ms')
    print()

In [ ]:
# Cell 9 — Download results
from google.colab import files
files.download('/content/trained_predictions.jsonl')
files.download('/content/prompt_only_predictions.jsonl')